# Operational Benchmark Estimation

This notebook documents the active estimation benchmark layer only. Historical enrollment-only and site-only benchmark artifacts, builders, checkers, and runtime utilities have been removed. The active runtime source is `src/operational_benchmarks.py`, with one builder and one checker:

```bash
python scripts/build_operational_benchmarks.py
python scripts/check_operational_benchmarks.py
```

The production UI reads `frontend/data/operational_benchmarks_v1.csv` for Planned Enrollment and Planned Site Count metadata/defaulting. The Excel companion `frontend/data/operational_benchmarks_v1.xlsx` is for analyst inspection only.

## Operational Benchmark Rules Reference

The active estimation layer uses one combined artifact: `frontend/data/operational_benchmarks_v1.csv`, built by `scripts/build_operational_benchmarks.py` and read at runtime by `src/operational_benchmarks.py`. The app also ships `operational_benchmarks_v1_report.json` for build summary and `operational_benchmarks_v1.xlsx` for human inspection only.

**Source populations**

- Enrollment percentiles use completed trials with positive `ACTUAL` enrollment.
- Site-count percentiles use completed trials with positive `number_of_facilities`.
- Patients-per-site percentiles use completed trials with positive `ACTUAL` enrollment and positive `number_of_facilities`, with `patients_per_site = enrollment / number_of_facilities`.
- `number_of_facilities` is a registry-derived facility-count proxy, not true planned sites or true activated sites.
- Percentiles are deterministic historical benchmarks, not an ML model, and do not enter XGBoost, SHAP, `/predict`, Completion Score, calibration, taxonomy, or prediction payloads.

**Cohort hierarchy**

Runtime lookup tries the strongest valid clinical cohort in this order:

1. `phase_indication_rare`: phase + indication + rare flag
2. `phase_ta_rare`: phase + therapeutic area + rare flag
3. `phase_ta`: phase + therapeutic area
4. `phase_only`: phase only

Invalid placeholder values cannot become specific cohorts:

- Indication id `0` or missing disables indication-level cohorts only.
- TA values `OTHER/UNCLASSIFIED`, `UNCLASSIFIED`, `UNKNOWN`, `OTHER`, or missing disable TA-level cohorts only.
- If indication is valid but TA is invalid, indication-level lookup is still allowed.
- Unknown/unclassified modality never creates or selects a modality-refinement row.

**Support thresholds**

- `n >= 50`: confident evidence.
- `30 <= n < 50`: usable low-confidence evidence for clinical fallback rows.
- `n < 30`: too sparse for that metric; fallback is required when possible.
- A selected clinical row is kept if at least one relevant operational metric is confident; weaker metrics on the same row remain flagged low confidence.
- Modality refinement and non-vaccine Infections fallback require `n >= 50` for the refined metric.

**Modality and Infections rules**

- Same-level modality refinement applies only to `enrollment` and `patients_per_site`, never raw `site_count`.
- Modality refinement is same-level only: it refines the already selected clinical level and never jumps to `phase + modality` or `phase_only_modality`.
- If modality refinement is unavailable, Infections trials with modality not equal to `VACCINE` can use non-vaccine Infections fallback rows for `enrollment` and `patients_per_site` only.
- Non-vaccine Infections means `TA = INFECTIONS` excluding `VACCINE`; unknown modality can contribute to that broad non-vaccine pool but cannot become an `UNKNOWN` modality row.
- Vaccine Infections trials can use vaccine modality refinement when supported and never use non-vaccine fallback.
- Raw `site_count` always remains clinical-only: no modality refinement and no non-vaccine Infections fallback.

**Defaulting and use in Simulation Mode**

- Planned Enrollment first uses a positive planned/estimated value when available.
- Completed trials without planned/estimated enrollment may use final observed enrollment.
- Non-completed trials without planned/estimated enrollment use `max(observed_lower_bound, enrollment_p50)` when available.
- Completed trials with positive `number_of_facilities` initialize Planned Sites from completed registry facility-count proxy.
- Non-completed trials treat positive `number_of_facilities` as current registry facility-count proxy lower-bound/context.
- Non-completed Planned Sites default to `max(current_registry_facility_count_proxy, planned_enrollment / patients_per_site_p50)` when patients-per-site P50 is available.
- Pure `site_count_p50` is fallback/reference only when the enrollment-coherent patients-per-site candidate cannot be calculated.
- User edits become scenario assumptions and update operational metadata without changing XGBoost score unless model-facing trial features also change.

**Classification bands**

- value `< p25`: `below_benchmark`
- `p25 <= value <= p75`: `typical`
- `p75 < value <= p90`: `ambitious`
- value `> p90`: `above_benchmark_high`


In [13]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "frontend" / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
ARTIFACT_PATH = PROJECT_ROOT / "frontend" / "data" / "operational_benchmarks_v1.csv"
REPORT_PATH = PROJECT_ROOT / "frontend" / "data" / "operational_benchmarks_v1_report.json"
EXCEL_PATH = PROJECT_ROOT / "frontend" / "data" / "operational_benchmarks_v1.xlsx"

artifact = pd.read_csv(ARTIFACT_PATH)
artifact.shape

(4006, 30)

## Artifact Structure

Each row is a benchmark cohort. Cohort keys describe the matching level: phase, indication, therapeutic area, rare flag, and optional modality/non-vaccine Infections refinement. Metric groups contain sample size and percentiles for:

- `enrollment_*`
- `site_count_*`
- `patients_per_site_*`

The runtime uses `p25`, `p50`, `p75`, and `p90` to classify operational assumptions as below benchmark, typical, ambitious, or above high benchmark.

In [14]:
artifact.columns.tolist()

['benchmark_version',
 'source_data_version',
 'benchmark_key',
 'phase',
 'gbd_cause_id_3_ml',
 'therapeutic_area',
 'rare_disease_flag',
 'therapeutic_modality',
 'benchmark_level_used',
 'enrollment_n',
 'enrollment_p25',
 'enrollment_p50',
 'enrollment_p75',
 'enrollment_p90',
 'enrollment_low_confidence_flag',
 'site_count_n',
 'site_count_p25',
 'site_count_p50',
 'site_count_p75',
 'site_count_p90',
 'site_count_low_confidence_flag',
 'patients_per_site_n',
 'patients_per_site_p25',
 'patients_per_site_p50',
 'patients_per_site_p75',
 'patients_per_site_p90',
 'patients_per_site_low_confidence_flag',
 'created_at',
 'outlier_policy',
 'calibration_notes']

In [15]:
artifact['benchmark_level_used'].value_counts().sort_index()

benchmark_level_used
phase_indication_rare                            656
phase_indication_rare_modality                  1948
phase_indication_rare_non_vaccine_infections      81
phase_only                                         4
phase_ta                                          72
phase_ta_modality                                440
phase_ta_non_vaccine_infections                    4
phase_ta_rare                                    133
phase_ta_rare_modality                           660
phase_ta_rare_non_vaccine_infections               8
Name: count, dtype: int64

## Runtime Lookup Contract

The runtime first tries clinical specificity:

1. `phase_indication_rare`
2. `phase_ta_rare`
3. `phase_ta`
4. `phase_only`

Invalid indication disables only indication-level cohorts. Invalid/unclassified TA disables only TA-level cohorts. Unknown/unclassified modality cannot form a modality-refinement row.

For enrollment and patients-per-site only, the runtime may refine the selected clinical row with same-level modality when `n >= 50`. For non-vaccine Infections, if modality refinement is unavailable, it may use Infections rows excluding vaccines, also requiring `n >= 50`. Raw site-count remains clinical-only.

In [16]:
import importlib
import src.operational_benchmarks as opb

opb = importlib.reload(opb)
benchmarks = opb.load_operational_benchmarks(ARTIFACT_PATH)
example_snapshot = {
    "phase": "PHASE3",
    "gbd_cause_id_3_ml": 302,
    "therapeutic_area": "INFECTIONS",
    "is_rare_disease_ml": 0,
    "therapeutic_modality_ui": "VACCINE",
}

{
    metric: opb.lookup_operational_benchmark(example_snapshot, benchmarks, metric_prefix=metric)[
        ["benchmark_level_used", f"{metric}_n", f"{metric}_p50", f"{metric}_low_confidence_flag"]
    ].to_dict()
    for metric in ("enrollment", "site_count", "patients_per_site")
}

{'enrollment': {'benchmark_level_used': 'phase_ta_rare_modality',
  'enrollment_n': 691,
  'enrollment_p50': 750.0,
  'enrollment_low_confidence_flag': False},
 'site_count': {'benchmark_level_used': 'phase_ta_rare',
  'site_count_n': 1003,
  'site_count_p50': 14.0,
  'site_count_low_confidence_flag': False},
 'patients_per_site': {'benchmark_level_used': 'phase_ta_rare_modality',
  'patients_per_site_n': 647,
  'patients_per_site_p50': 75.0,
  'patients_per_site_low_confidence_flag': False}}

## Defaulting Rules

Planned Enrollment uses a planned/estimated value when available. For non-completed trials without a planned/estimated value, current observed enrollment is treated as a lower bound and the default is:

```text
max(observed_lower_bound, enrollment_p50)
```

Planned Sites uses completed registry facility count for completed trials. For non-completed trials, current registry facility count is lower-bound context and the default is:

```text
max(current_registry_facility_count_proxy, planned_enrollment / patients_per_site_p50)
```

Pure `site_count_p50` is fallback/reference when patients-per-site cannot be calculated.

In [17]:
opb.planned_enrollment_default_from_operational_benchmark(
    example_snapshot,
    observed_lower_bound=150,
    artifact=benchmarks,
)

{'value': 750,
 'source': 'model_default',
 'observed_lower_bound': 150.0,
 'enrollment_benchmark_p50': 750.0,
 'operational_benchmark_snapshot_id': 'operational_benchmarks_v1:0a97519bd78f561a:phase_ta_rare_modality|phase=PHASE3|ta=INFECTIONS|rare=0|modality=VACCINE'}

In [18]:
opb.planned_sites_default_from_operational_benchmark(
    example_snapshot,
    planned_enrollment=500,
    current_registry_facility_count_proxy=12,
    overall_status="RECRUITING",
    artifact=benchmarks,
)

{'value': 12,
 'source': 'current_registry_facility_count_proxy',
 'site_default_basis': 'current_registry_facility_count_proxy',
 'current_registry_facility_count_proxy': 12.0,
 'site_count_benchmark_p50': 14.0,
 'patients_per_site_p50': 75.0,
 'patients_per_site_benchmark_level_used': 'phase_ta_rare_modality',
 'patients_per_site_n': 647,
 'patients_per_site_low_confidence_flag': False,
 'enrollment_coherent_site_candidate': 6.666666666666667,
 'operational_benchmark_snapshot_id': 'operational_benchmarks_v1:0a97519bd78f561a:phase_ta_rare_modality|phase=PHASE3|ta=INFECTIONS|rare=0|modality=VACCINE'}

## Rule Examples

These examples exercise the operational fallback rules directly. They are intentionally small and deterministic so changes in lookup behavior are easy to spot.

In [19]:
def summarize_lookup(label, snapshot):
    rows = []
    for metric in ("enrollment", "site_count", "patients_per_site"):
        row = opb.lookup_operational_benchmark(snapshot, benchmarks, metric_prefix=metric)
        rows.append({
            "example": label,
            "metric": metric,
            "level": None if row is None else row.get("benchmark_level_used"),
            "key": None if row is None else row.get("benchmark_key"),
            "n": None if row is None else int(row.get(f"{metric}_n")),
            "p50": None if row is None else row.get(f"{metric}_p50"),
            "low_confidence": None if row is None else bool(row.get(f"{metric}_low_confidence_flag")),
        })
    return rows

rule_examples = [
    (
        "invalid indication -> TA fallback",
        {
            "phase": "PHASE3",
            "gbd_cause_id_3_ml": 0,
            "therapeutic_area": "ONCOLOGY",
            "is_rare_disease_ml": 0,
            "therapeutic_modality_ui": "SMALL MOLECULE",
        },
    ),
    (
        "invalid TA -> keep indication",
        {
            "phase": "PHASE3",
            "gbd_cause_id_3_ml": 426,
            "therapeutic_area": "UNCLASSIFIED",
            "is_rare_disease_ml": 0,
            "therapeutic_modality_ui": "SMALL MOLECULE",
        },
    ),
    (
        "unknown modality -> no modality refinement",
        {
            "phase": "PHASE3",
            "gbd_cause_id_3_ml": 426,
            "therapeutic_area": "ONCOLOGY",
            "is_rare_disease_ml": 0,
            "therapeutic_modality_ui": "UNKNOWN",
        },
    ),
    (
        "vaccine Infections -> vaccine refinement",
        {
            "phase": "PHASE3",
            "gbd_cause_id_3_ml": 302,
            "therapeutic_area": "INFECTIONS",
            "is_rare_disease_ml": 0,
            "therapeutic_modality_ui": "VACCINE",
        },
    ),
    (
        "non-vaccine Infections -> non-vaccine fallback",
        {
            "phase": "PHASE3",
            "gbd_cause_id_3_ml": 302,
            "therapeutic_area": "INFECTIONS",
            "is_rare_disease_ml": 0,
            "therapeutic_modality_ui": "BIOLOGIC MAB",
        },
    ),
]

pd.DataFrame([row for label, snapshot in rule_examples for row in summarize_lookup(label, snapshot)])

,example,metric,level,key,n,p50,low_confidence
0,invalid indication -> TA fallback,enrollment,phase_ta_rare_modality,phase_ta_rare_modality|phase=PHASE3|ta=ONCOLOG...,395,413.00,False
1,invalid indication -> TA fallback,site_count,phase_ta_rare,phase_ta_rare|phase=PHASE3|ta=ONCOLOGY|rare=0,870,78.00,False
2,invalid indication -> TA fallback,patients_per_site,phase_ta_rare_modality,phase_ta_rare_modality|phase=PHASE3|ta=ONCOLOG...,391,5.00,False
3,invalid TA -> keep indication,enrollment,phase_indication_rare_modality,phase_indication_rare_modality|phase=PHASE3|in...,60,355.50,False
4,invalid TA -> keep indication,site_count,phase_indication_rare,phase_indication_rare|phase=PHASE3|indication=...,165,78.00,False
5,invalid TA -> keep indication,patients_per_site,phase_indication_rare_modality,phase_indication_rare_modality|phase=PHASE3|in...,60,7.12,False
6,unknown modality -> no modality refinement,enrollment,phase_indication_rare,phase_indication_rare|phase=PHASE3|indication=...,170,453.00,False
7,unknown modality -> no modality refinement,site_count,phase_indication_rare,phase_indication_rare|phase=PHASE3|indication=...,165,78.00,False
8,unknown modality -> no modality refinement,patients_per_site,phase_indication_rare,phase_indication_rare|phase=PHASE3|indication=...,163,5.53,False
9,vaccine Infections -> vaccine refinement,enrollment,phase_ta_rare_modality,phase_ta_rare_modality|phase=PHASE3|ta=INFECTI...,691,750.00,False


## Planned Sites Default Where Enrollment-Coherent Candidate Wins

This example uses a high planned enrollment and low current registry facility-count proxy. The default should choose `planned_enrollment / patients_per_site_p50` because it is larger than the current proxy.

In [20]:
site_win_snapshot = {
    "phase": "PHASE3",
    "gbd_cause_id_3_ml": 302,
    "therapeutic_area": "INFECTIONS",
    "is_rare_disease_ml": 0,
    "therapeutic_modality_ui": "VACCINE",
}

opb.planned_sites_default_from_operational_benchmark(
    site_win_snapshot,
    planned_enrollment=7500,
    current_registry_facility_count_proxy=12,
    overall_status="RECRUITING",
    artifact=benchmarks,
)

{'value': 100,
 'source': 'enrollment_coherent_benchmark_default',
 'site_default_basis': 'enrollment_coherent_benchmark_default',
 'current_registry_facility_count_proxy': 12.0,
 'site_count_benchmark_p50': 14.0,
 'patients_per_site_p50': 75.0,
 'patients_per_site_benchmark_level_used': 'phase_ta_rare_modality',
 'patients_per_site_n': 647,
 'patients_per_site_low_confidence_flag': False,
 'enrollment_coherent_site_candidate': 100.0,
 'operational_benchmark_snapshot_id': 'operational_benchmarks_v1:0a97519bd78f561a:phase_ta_rare_modality|phase=PHASE3|ta=INFECTIONS|rare=0|modality=VACCINE'}

## Validation

The single checker validates schema, fallback behavior, modality/non-vaccine rules, registry-wide coverage, site defaulting safety, and model-boundary safeguards. Run it before relying on the artifact after any rebuild.